# Offline GPU Embedding for Multilingual RAG Exports

Notebook này chạy trực tiếp trên Kaggle và in progress/ETA ngay trong output cell.

Output tạo ra ở `/kaggle/working/`:
- `embeddings.npy`
- `chunk_ids.json`
- `embedding_manifest.json`


In [ ]:
!pip install -q -U sentence-transformers
print('Done.')


In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import numpy as np
import torch

MODEL_NAME = 'BAAI/bge-m3'
BATCH_SIZE = 64
PROGRESS_EVERY = 50
NORMALIZE_EMBEDDINGS = True
MAX_ROWS = 0  # set > 0 for quick test

# Đổi path này nếu anh gắn dataset vào chỗ khác trên Kaggle
INPUT_PATH = Path('/kaggle/input/datasets/sonjpro/embedding-batch-3/chunk_texts_for_embed.jsonl')

def _log(msg: str) -> None:
    now = time.strftime('%H:%M:%S')
    print(f'[{now}] {msg}', flush=True)

def _gpu_mem_gb() -> str:
    if not torch.cuda.is_available():
        return 'cpu'
    used = torch.cuda.memory_allocated(0) / 1024 / 1024 / 1024
    reserved = torch.cuda.memory_reserved(0) / 1024 / 1024 / 1024
    total = torch.cuda.get_device_properties(0).total_memory / 1024 / 1024 / 1024
    return f'{used:.1f}/{reserved:.1f}/{total:.1f} GB'

if not INPUT_PATH.exists():
    candidates = []
    for root, _, files in os.walk('/kaggle/input'):
        for name in files:
            if name == 'chunk_texts_for_embed.jsonl':
                candidates.append(Path(root) / name)
    if not candidates:
        raise FileNotFoundError(f'Input path not found: {INPUT_PATH}')
    candidates.sort()
    INPUT_PATH = candidates[0]

_log(f'Input: {INPUT_PATH}')

ids = []
texts = []
seen = set()
with open(INPUT_PATH, 'r', encoding='utf-8') as fh:
    for line_no, raw in enumerate(fh, start=1):
        if not raw.strip():
            continue
        rec = json.loads(raw)
        chunk_id = str(rec['id'])
        if chunk_id in seen:
            raise ValueError(f'Duplicate chunk id at line {line_no}: {chunk_id}')
        seen.add(chunk_id)
        ids.append(chunk_id)
        texts.append(str(rec['text']))
        if MAX_ROWS and len(ids) >= MAX_ROWS:
            break

if not ids:
    raise ValueError('Input JSONL contained no usable rows')

avg_chars = sum(len(t) for t in texts) / max(1, len(texts))
_log(f'Loaded {len(texts)} chunks | avg_chars={avg_chars:.0f}')
_log(f'Sample ID: {ids[0]}')
_log(f"Sample text[:160]: {texts[0][:160].replace(chr(10), ' ')}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
_log(f'Device: {device}')
if device == 'cuda':
    _log(f'GPU: {torch.cuda.get_device_name(0)} | mem={_gpu_mem_gb()}')


In [ ]:
from sentence_transformers import SentenceTransformer

_log(f'Loading model: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME, device=device)
dim = model.get_sentence_embedding_dimension()
_log(f'Model loaded | embedding_dim={dim}')

_log('Warmup...')
_ = model.encode(['warmup'], normalize_embeddings=NORMALIZE_EMBEDDINGS, show_progress_bar=False)
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()
    _log(f'Warmup done | gpu_mem={_gpu_mem_gb()}')
else:
    _log('Warmup done')

t0 = time.time()
all_emb = []
total = len(texts)
total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
_log(f'Start embedding | total_chunks={total} | batch_size={BATCH_SIZE} | total_batches={total_batches}')

for batch_index, start in enumerate(range(0, total, BATCH_SIZE), start=1):
    batch = texts[start:start + BATCH_SIZE]
    vecs = model.encode(
        batch,
        batch_size=BATCH_SIZE,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    all_emb.append(vecs)

    should_log = (
        batch_index == 1
        or batch_index == total_batches
        or batch_index % max(1, PROGRESS_EVERY) == 0
    )
    if should_log:
        done = min(start + len(batch), total)
        elapsed = time.time() - t0
        rate = done / elapsed if elapsed > 0 else 0.0
        remaining = total - done
        eta_s = remaining / rate if rate > 0 else 0.0
        eta_m = eta_s / 60
        pct = done / total * 100
        msg = (
            f'Progress {done}/{total} ({pct:.1f}%) | '
            f'batch={batch_index}/{total_batches} | '
            f'elapsed={elapsed/60:.1f}m | rate={rate:.0f} vec/s | ETA={eta_m:.1f}m'
        )
        if device == 'cuda':
            msg += f' | gpu_mem={_gpu_mem_gb()}'
        _log(msg)

elapsed = time.time() - t0
emb_array = np.vstack(all_emb).astype(np.float32)
norms = np.linalg.norm(emb_array[: min(5, len(emb_array))], axis=1)
_log(
    f'Embedding complete | shape={emb_array.shape} | dtype={emb_array.dtype} | '
    f'time={elapsed/60:.1f}m | avg_rate={emb_array.shape[0]/elapsed:.0f} vec/s'
)
_log(f'Sample norms: {norms}')


In [ ]:
out_dir = Path('/kaggle/working')
emb_path = out_dir / 'embeddings.npy'
ids_path = out_dir / 'chunk_ids.json'
manifest_path = out_dir / 'embedding_manifest.json'

_log(f'Saving embeddings -> {emb_path}')
np.save(emb_path, emb_array)
_log(f'Saving chunk ids -> {ids_path}')
with open(ids_path, 'w', encoding='utf-8') as fh:
    json.dump(ids, fh, ensure_ascii=False)

manifest = {
    'model_name': MODEL_NAME,
    'batch_size': BATCH_SIZE,
    'normalize_embeddings': NORMALIZE_EMBEDDINGS,
    'input_path': str(INPUT_PATH),
    'chunk_count': len(ids),
    'vector_shape': list(emb_array.shape),
    'dtype': str(emb_array.dtype),
    'elapsed_seconds': round(elapsed, 2),
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
_log(f'Saved manifest -> {manifest_path}')
_log(f'Done | embeddings_mb={emb_array.nbytes/1024/1024:.1f}')
print('Download embeddings.npy and chunk_ids.json from the Output tab.')
